In [1]:
import pandas as pd
import numpy as np
import ast
import os
import matplotlib.pyplot as plt
import bayesian_analysis as ba

rng = np.random.default_rng(66)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


### Load data

In [2]:
folder = "C:\\Users\\shirl\\OneDriveCloudTemp\\G25Y1LCK\\Data_29-05-26\\Data_29-05-26"
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
challenge_completions = pd.read_csv(os.path.join(folder, 'challenge_completions.csv'), index_col=0)
participant_conditions = pd.read_csv(os.path.join(folder, 'participant_conditions.csv'), index_col=0)
postquestions = pd.read_csv(os.path.join(folder, 'postquestions.csv'), index_col=0)
prequestions = pd.read_csv(os.path.join(folder, 'prequestions.csv'), index_col=0)
presented_challenges = pd.read_csv(os.path.join(folder, 'presented_challenges.csv'), index_col=0)
challenges_sets = pd.read_csv(os.path.join(folder, 'challenges_sets.csv'), index_col=0)

photo_df = pd.read_csv(os.path.join(folder, 'foto_finalQuestionnaire.csv'), index_col=0)

age_df = pd.read_csv(os.path.join(folder, 'dem_age.csv'), index_col=0)
edu_current_df = pd.read_csv(os.path.join(folder, 'dem_current_edu.csv'), index_col=0)
edu_finished_df = pd.read_csv(os.path.join(folder, 'dem_finished_edu.csv'), index_col=0)
gender_df = pd.read_csv(os.path.join(folder, 'dem_gender.csv'), index_col=0)

challenge_info = pd.read_csv(os.path.join(data_folder, 'challenges_real.csv'), delimiter=';')

### Check and remove duplicates

In [3]:
# Each participant should have one row per day in the postquestions, with consistent values for 'Likedness', 'difficulty', 'time_spent', and 'usefulness' across rows with the same 'random_id' and 'day_number'. We will check for any inconsistencies in these columns.
keys = ['random_id', 'day_number']
postquestions_columns = ['random_id', 'day_number', 'Likedness', 'difficulty', 'time_spent', 'usefulness']

# Only keep relevant columns
postquestions = postquestions[postquestions_columns] 
mismatched = postquestions.groupby(keys).filter(lambda x: x.nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")
print(mismatched.sort_values(by=keys))

# We keep the last entry for each duplicate key combination, assuming it is the most recent and therefore the most accurate reflection of the participant's experience for that day.
postquestions = postquestions.drop_duplicates(subset=keys, keep='last')

Found 8 rows with inconsistent ratings across keys.
       random_id  day_number  Likedness  difficulty  time_spent  usefulness
2037  3961265261          20          7           1           1           5
2038  3961265261          20          7           1           1           4
3023  4201595922          10          3           0           1           2
3024  4201595922          10          3           0           1           3
3880  6549566506          10          5           3          10           6
3881  6549566506          10          5           3          10           4
3369  9975681399          25          4           0           2           4
3370  9975681399          25          6           0           2           4


In [4]:
keys = ['random_id', 'day_number']
presented_challenges_columns = ['random_id', 'day_number', 'challenge_id']
# presented_challenges = presented_challenges[presented_challenges_columns]

mismatched = presented_challenges.groupby(keys).filter(lambda x: x[presented_challenges_columns].nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")
unique_mismatched_keys = mismatched[keys].drop_duplicates()
presented_challenges = presented_challenges.drop_duplicates(subset=keys, keep='last')

Found 5702 rows with inconsistent ratings across keys.


Challenge completions should not contain any duplicates where the same participant has multiple completed challenge_ids for the same day


In [5]:
keys = ['random_id', 'day_number']
challenge_completions_columns = ['random_id', 'day_number', 'challenge_id', 'challenge_category']
challenge_completions = challenge_completions[challenge_completions_columns]

mismatched = challenge_completions.groupby(keys).filter(lambda x: x.nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")
print(mismatched.sort_values(by=keys))

Found 0 rows with inconsistent ratings across keys.
Empty DataFrame
Columns: [random_id, day_number, challenge_id, challenge_category]
Index: []


The number of presented challenges should be the same as the number of filled in prequestions

In [6]:
print(f"Number of presented challenges: {len(presented_challenges)}")
print(f"Number of prequestions: {len(prequestions)}")

merged = pd.merge(presented_challenges, prequestions, on=['random_id', 'day_number'], how='outer', indicator=True)
mismatched = merged[merged['_merge'] != 'both']
print(f"Found {len(mismatched)} rows with mismatched presented challenges and prequestions.")
display(mismatched.sort_values(by=['random_id', 'day_number']))

Number of presented challenges: 5797
Number of prequestions: 5801
Found 4 rows with mismatched presented challenges and prequestions.


,random_id,date_x,time_x,day_number,challenge_category,challenge_id,created_at_x,date_y,time_y,TIR,MOT,GOOD,TIME_Q,PAY,created_at_y,_merge
1331,2688238464,NaN,NaN,17,NaN,NaN,NaN,2026-05-29,12:36:25,2,2,4,1,3,2026-05-29 10:36:24,right_only
2691,4707708259,NaN,NaN,18,NaN,NaN,NaN,2026-05-29,12:31:58,5,4,5,4,4,2026-05-29 10:31:58,right_only
3255,5526027943,NaN,NaN,19,NaN,NaN,NaN,2026-05-29,11:38:24,2,5,3,6,5,2026-05-29 09:38:24,right_only
5634,9709689980,NaN,NaN,15,NaN,NaN,NaN,2026-05-29,11:43:06,6,2,2,5,5,2026-05-29 09:43:05,right_only


### Extract completed challenge information
We merge the completed challenges with the postquestions to obtain the ratings for the challenges

In [7]:
keys = ['random_id', 'day_number', 'challenge_id']
completed_challenges = challenge_completions[['random_id', 'day_number', 'challenge_id', 'challenge_category']].copy()

mismatched = completed_challenges[completed_challenges.duplicated(keys, keep=False)]
print(f"Number of duplicate challenge entries: {len(mismatched)}")
completed_challenges = completed_challenges.drop_duplicates(subset=keys, keep='last')

completed_challenges = completed_challenges.merge(postquestions, on=['random_id', 'day_number'], how='left')
print("Number of challenge ratings:", len(completed_challenges))
completed_challenges.head()

difficulty_to_score = {
    "Makkelijk": 0.33,
    "Gemiddeld": 0.66,
    "Moeilijk": 1.0
}
challenge_categories = completed_challenges[['challenge_id', 'challenge_category']].drop_duplicates().set_index('challenge_id')['challenge_category']
cat_to_id = {c: int(i) for i, c in enumerate(sorted(challenge_categories.unique()))}

# TODO: Use all data or only from the ones we use in the samples as well? I.e. filter on conditions
mean_challenge_scores = completed_challenges.groupby('challenge_id')[['Likedness', 'difficulty', 'time_spent', 'usefulness']].mean().rename(columns={'Likedness': 'likedness'})

challenge_info['expert_score'] = challenge_info['Moeilijkheidsgraad'].map(difficulty_to_score)
challenge_info = challenge_info.merge(mean_challenge_scores, on='challenge_id', how='outer')
challenge_info = challenge_info.merge(challenge_categories.rename('category'), on='challenge_id', how='left')
challenge_info['category_id'] = challenge_info['category'].map(cat_to_id)
challenge_info.rename(columns={'Likedness': 'likedness'}, inplace=True)
challenge_info.to_csv(os.path.join(data_folder, 'challenge_info.csv'), index=False)
challenge_info.head()

Number of duplicate challenge entries: 807
Number of challenge ratings: 3785


,challenge_id,Type,Moeilijkheidsgraad,expert_score,likedness,difficulty,time_spent,usefulness,category,category_id
0,AC1,Photo,Moeilijk,1.00,7.111111,2.444444,13.555556,4.592593,acceptance,0
1,AC10,Open,Moeilijk,1.00,5.687500,3.609375,2.937500,4.359375,acceptance,0
2,AC11,Open,Moeilijk,1.00,5.864865,2.108108,3.594595,4.324324,acceptance,0
3,AC12,Open,Gemiddeld,0.66,5.884058,2.318841,1.898551,4.115942,acceptance,0
4,AC13,Photo,Moeilijk,1.00,6.571429,3.928571,6.571429,4.285714,acceptance,0


### Extract relevant data

In [8]:
def get_condition(row):
    day = row['day_number']
    order_list = row['within_order']
    if day <= 5:
        return 'w0'
    idx = ((day - 1) // 5) - 1
    return order_list[idx]

In [9]:
prequestions_cols = ['random_id', 'day_number', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY']
df_samples = prequestions[prequestions_cols].copy()

# Merge participant conditions to get between_condition and within_order
df_samples = df_samples.merge(participant_conditions[['random_id', 'between_condition', 'within_order']], on='random_id', how='left')
df_samples['within_order'] = df_samples['within_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df_samples['within_condition'] = df_samples.apply(lambda row: get_condition(row), axis=1)

# Save initial states distribution for Day 1 for all users, regardless of their condition
initial_states = df_samples[df_samples['day_number'] == 1]
initial_states[['random_id', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY']].to_csv(os.path.join(data_folder, 'initial_states_distribution.csv'), index=False)

# Merge with recommended challenges to get challenge_id for each sample
recommended_challenges = challenges_sets[['random_id', 'day_number', 'recommended_challenge_id']].copy()
df_samples = df_samples.merge(recommended_challenges, on=['random_id', 'day_number'], how='left')

# Mark everything in the completed df as 1
completed_challenges['completed'] = 1

# Merge challenge completions
df_samples = df_samples.merge(
    completed_challenges[['random_id', 'day_number', 'challenge_id', 'completed', 'Likedness', 'difficulty', 'time_spent', 'usefulness']], 
    on=['random_id', 'day_number'], 
    how='left'
)

deviations = df_samples[
    df_samples['challenge_id'].notna() &
    (df_samples['challenge_id'] != df_samples['recommended_challenge_id'])
]
print(f"Number of samples where the presented challenge does not match the recommended challenge: {len(deviations)}")
assert len(deviations[deviations['within_condition'].isin(['w0','w1', 'w2'])]) == 0, "There are samples in the non-agency conditions where the presented challenge does not match the recommended challenge."

# Fill not completed challenges accordingly
df_samples['completed'] = df_samples['completed'].fillna(0).astype(int)
df_samples['challenge_id'] = df_samples['challenge_id'].fillna(df_samples['recommended_challenge_id'])

# Set postquestions to NaN for non-completed challenges
post_cols = ['Likedness', 'difficulty', 'time_spent', 'usefulness']
for col in post_cols:
    df_samples.loc[df_samples['completed'] == 0, col] = np.nan

df_samples = df_samples.sort_values(by=['random_id', 'day_number'])
df_samples['TIR_next'] = df_samples.groupby('random_id')['TIR'].shift(-1)
df_samples['MOT_next'] = df_samples.groupby('random_id')['MOT'].shift(-1)
df_samples['GOOD_next'] = df_samples.groupby('random_id')['GOOD'].shift(-1)
df_samples['TIME_Q_next'] = df_samples.groupby('random_id')['TIME_Q'].shift(-1)
df_samples['PAY_next'] = df_samples.groupby('random_id')['PAY'].shift(-1)

# remove rows where next state is NaN (i.e., last day for each user)
# df_samples = df_samples.dropna(subset=['TIR_next', 'MOT_next', 'GOOD_next', 'TIME_Q_next', 'PAY_next'])
df_samples = df_samples.rename(columns={"Likedness" : "likedness", "random_id": "user_id"})

final_columns = [
        'user_id', 'day_number', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY',
        'TIR_next', 'MOT_next', 'GOOD_next', 'TIME_Q_next', 'PAY_next', 'recommended_challenge_id',
        'challenge_id', 'completed', 'likedness', 'difficulty', 'time_spent', 'usefulness', 'within_condition'
    ]
df_samples = df_samples[final_columns]

# Remove rows where challenge_id is NaN, as these represent the day where the data was pulled and no challenge was presented yet
df_samples.dropna(subset=['challenge_id'], inplace=True)

# Merge challenge information into samples
df_samples = df_samples.merge(challenge_info[['challenge_id', 'category', 'category_id', 'Type', 'expert_score']], on='challenge_id', how='left')

print("Total number of samples:", len(df_samples))
print("Total number of users:", df_samples['user_id'].nunique())
df_samples.head()

Number of samples where the presented challenge does not match the recommended challenge: 537
Total number of samples: 5797
Total number of users: 318


,user_id,day_number,TIR,MOT,GOOD,TIME_Q,PAY,TIR_next,MOT_next,GOOD_next,...,completed,likedness,difficulty,time_spent,usefulness,within_condition,category,category_id,Type,expert_score
0,1002441714,1,1,4,5,4,4,4.0,4.0,5.0,...,0,NaN,NaN,NaN,NaN,w0,acceptance,0,Photo,1.00
1,1002441714,2,4,4,5,5,4,4.0,4.0,5.0,...,1,7.0,7.0,1.0,2.0,w0,problem_solving,2,Quiz,0.66
2,1002441714,3,4,4,5,4,4,4.0,4.0,5.0,...,1,1.0,8.0,0.0,1.0,w0,social_support,3,Open,1.00
3,1002441714,4,4,4,5,5,4,2.0,4.0,4.0,...,1,6.0,6.0,60.0,4.0,w0,social_support,3,Photo,0.66
4,1002441714,5,2,4,4,4,4,6.0,4.0,4.0,...,1,7.0,5.0,1.0,4.0,w0,acceptance,0,Photo,1.00


### Save

In [10]:
# Only keep samples from first 5 days (w0), and from control (w1) and explanation (w2) conditions
# TODO: also filter on between condition?
conditions_to_keep = ['w0', 'w1', 'w2']
df_final = df_samples[df_samples['within_condition'].isin(conditions_to_keep)].copy()
# print number of samples per condition

df_agency = df_samples[df_samples['within_condition'].isin(['w3', 'w4'])].copy() 

print("Final number of samples (filtered):", len(df_final))
print("Final number of users:", df_final['user_id'].nunique())

df_final.to_csv(os.path.join(data_folder, 'processed_samples.csv'), index=False)
df_final.head()

df_agency.to_csv(os.path.join(data_folder, 'agency_samples.csv'), index=False)
df_agency.head()

Final number of samples (filtered): 3632
Final number of users: 318


,user_id,day_number,TIR,MOT,GOOD,TIME_Q,PAY,TIR_next,MOT_next,GOOD_next,...,completed,likedness,difficulty,time_spent,usefulness,within_condition,category,category_id,Type,expert_score
15,1002441714,16,3,1,2,2,1,2.0,1.0,2.0,...,1,7.0,1.0,1.0,5.0,w4,distraction,1,Open,0.33
16,1002441714,17,2,1,2,2,1,4.0,1.0,1.0,...,1,7.0,2.0,1.0,4.0,w4,problem_solving,2,Quiz,0.66
17,1002441714,18,4,1,1,2,1,4.0,1.0,1.0,...,1,0.0,0.0,8.0,1.0,w4,acceptance,0,Open,0.66
18,1002441714,19,4,1,1,2,1,6.0,1.0,1.0,...,0,NaN,NaN,NaN,NaN,w4,distraction,1,Photo,0.66
19,1002441714,20,6,1,1,1,1,4.0,1.0,1.0,...,1,6.0,1.0,1.0,1.0,w4,problem_solving,2,Quiz,0.66


In [11]:
df_samples.groupby('Type')['completed'].mean()

Type
Open     0.746364
Photo    0.364610
Quiz     0.968966
Name: completed, dtype: float64

## Adjust incompleted photo challenges
Due to a bug in the application, participants had trouble uploading photos for these type of challenges. There was a workaround, however for some people this still did not work. Therefore, some were incorrectly marked as not completed, and there is no postquestion data on these.

In [12]:
mean_vals_photo = df_samples[(df_samples['Type'] == 'Photo') & (df_samples['completed'] == 1)].groupby('category_id')[['likedness', 'difficulty', 'usefulness', 'time_spent']].mean()
mean_vals_photo

,likedness,difficulty,usefulness,time_spent
category_id,,,,
0,5.962264,3.433962,3.886792,8.886792
1,5.816770,2.860248,3.618012,7.897516
2,5.154930,3.154930,3.859155,6.323944
3,6.547945,3.767123,4.178082,44.123288


In [13]:
# Aggregate in one pass
photo_counts = (
    df_samples[df_samples['Type'] == 'Photo']
    .groupby('user_id')['completed']
    .agg(
        photos_completed='sum',
        photos_not_completed=lambda x: (x == 0).sum()
    )
    .astype(int)
)

# Merge and fill in one step
photo_df_corrections = (
    photo_counts
    .merge(photo_df[['random_id', 'photo_challenge_n']], left_index=True, right_on='random_id', how='left')
    .fillna({'photo_challenge_n': 0})
    .assign(photo_challenge_n=lambda df: df['photo_challenge_n'].astype(int))
)

# Compute rates
total = photo_df_corrections['photos_completed'] + photo_df_corrections['photos_not_completed']

# for every user, find probability that the not-completed photo, was actually completed, i.e. photo_challenge_n / photos_not_completed, but only for users where photos_not_completed > 0 to avoid division by zero
photo_df_corrections['prob_photo_completed'] = np.where(
    photo_df_corrections['photos_not_completed'] > 0,
    np.minimum(photo_df_corrections['photo_challenge_n'], photo_df_corrections['photos_not_completed']) / total,
    0)

# for every user, adjust the samples according to the probability found above
# i.e. for every sample where Type is Photo and completed is 0, we set completed to 1 with the probability found above for that user
# additionally, impute mean values for likedness, difficulty, usefulness, and time_spent for the newly completed photos, 
df_samples_adjusted = df_samples.copy()

for idx, row in photo_df_corrections.iterrows():
    user_id = row['random_id']
    prob_completed = row['prob_photo_completed']
    
    # Get indices of samples for this user where Type is Photo and completed is 0
    mask = (df_samples_adjusted['user_id'] == user_id) & (df_samples_adjusted['Type'] == 'Photo') & (df_samples_adjusted['completed'] == 0)
    indices_to_adjust = df_samples_adjusted[mask].index
  
    # For each of these samples, we will set completed to 1 with the probability found above for that user
    for idx in indices_to_adjust:
        category_id = df_samples_adjusted.at[idx, 'category_id']
        if rng.random() < prob_completed:
            df_samples_adjusted.at[idx, 'completed'] = 1
            df_samples_adjusted.at[idx, 'likedness'] = mean_vals_photo.loc[category_id, 'likedness']
            df_samples_adjusted.at[idx, 'difficulty'] = mean_vals_photo.loc[category_id, 'difficulty']
            df_samples_adjusted.at[idx, 'usefulness'] = mean_vals_photo.loc[category_id, 'usefulness']
            df_samples_adjusted.at[idx, 'time_spent'] = mean_vals_photo.loc[category_id, 'time_spent']

df_samples_adjusted.groupby('Type')[['completed']].mean()

,completed
Type,
Open,0.746364
Photo,0.467254
Quiz,0.968966


In [14]:
df_samples_adjusted.to_csv(os.path.join(data_folder, 'processed_samples_adjusted_full.csv'), index=False)

conditions_to_keep = ['w0', 'w1', 'w2']
df_final_adjusted = df_samples_adjusted[df_samples_adjusted['within_condition'].isin(conditions_to_keep)].copy()

df_agency_adjusted = df_samples_adjusted[df_samples_adjusted['within_condition'].isin(['w3', 'w4'])].copy() 

print("Final number of samples (filtered):", len(df_final_adjusted))
print("Final number of users:", df_final_adjusted['user_id'].nunique())

df_final_adjusted.to_csv(os.path.join(data_folder, 'processed_samples_adjusted.csv'), index=False)
df_final_adjusted.head()

df_agency_adjusted.to_csv(os.path.join(data_folder, 'agency_samples_adjusted.csv'), index=False)
df_agency_adjusted.head()

Final number of samples (filtered): 3632
Final number of users: 318


,user_id,day_number,TIR,MOT,GOOD,TIME_Q,PAY,TIR_next,MOT_next,GOOD_next,...,completed,likedness,difficulty,time_spent,usefulness,within_condition,category,category_id,Type,expert_score
15,1002441714,16,3,1,2,2,1,2.0,1.0,2.0,...,1,7.0,1.0,1.0,5.0,w4,distraction,1,Open,0.33
16,1002441714,17,2,1,2,2,1,4.0,1.0,1.0,...,1,7.0,2.0,1.0,4.0,w4,problem_solving,2,Quiz,0.66
17,1002441714,18,4,1,1,2,1,4.0,1.0,1.0,...,1,0.0,0.0,8.0,1.0,w4,acceptance,0,Open,0.66
18,1002441714,19,4,1,1,2,1,6.0,1.0,1.0,...,0,NaN,NaN,NaN,NaN,w4,distraction,1,Photo,0.66
19,1002441714,20,6,1,1,1,1,4.0,1.0,1.0,...,1,6.0,1.0,1.0,1.0,w4,problem_solving,2,Quiz,0.66


### Inspect in agency conditions how often users deviate

In [15]:
agency_deviations = df_agency.copy()
agency_deviations['deviated'] = agency_deviations['challenge_id'] != agency_deviations['recommended_challenge_id']
print(f"Number of samples in agency conditions: {len(agency_deviations)}")
print(f"Number of samples in agency conditions where the presented challenge does not match the recommended challenge: {agency_deviations['deviated'].sum()}")
print(f"Completion rate in no agency conditions (w0, w1, w2): {df_final['completed'].mean():.2f}")
print(f"Completion rate in agency conditions (w3, w4): {df_agency['completed'].mean():.2f}")
agency_deviations.groupby('deviated')['completed'].mean()

deviation_prob = agency_deviations['deviated'].mean()
print(f"Probability of deviation in agency conditions: {deviation_prob:.2f}")

Number of samples in agency conditions: 2165
Number of samples in agency conditions where the presented challenge does not match the recommended challenge: 537
Completion rate in no agency conditions (w0, w1, w2): 0.64
Completion rate in agency conditions (w3, w4): 0.68
Probability of deviation in agency conditions: 0.25


### Compare adherence in agency vs. non-agency conditions

In [16]:
# Per-person completion rates
agency_rates = (
    df_agency_adjusted
    .groupby("user_id")["completed"]
    .mean()
)

control_rates = (
    df_final_adjusted
    .groupby("user_id")["completed"]
    .mean()
)

paired = pd.concat(
    [agency_rates, control_rates],
    axis=1,
    keys=["agency", "control"]
).dropna()

# Per-person differences
paired["diff"] = paired["agency"] - paired["control"]

diffs = paired["diff"].values

ba.paired_t_test(diffs)

              mean     sd  hdi_2.5%  hdi_97.5%  mcse_mean  mcse_sd  ess_bulk  \
improvement  0.038  0.013     0.013      0.062      0.000    0.000  2484.017   
effect_size  0.192  0.069     0.059      0.323      0.001    0.001  2248.721   

             ess_tail  r_hat  
improvement  2496.825  1.002  
effect_size  2282.183  1.002  
P(improvement > 0): 0.9985
P(Effect Size > 0.1): 0.9110
P(Effect Size > 0.2): 0.4407
P(Effect Size > 0.3): 0.0620


### Inspect data

#### Correlations

In [17]:
correlation_states = df_final[['TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY']].corr()
print(correlation_states)

             TIR       MOT      GOOD    TIME_Q       PAY
TIR     1.000000 -0.167875 -0.050017 -0.121962 -0.042782
MOT    -0.167875  1.000000  0.531044  0.459284  0.506574
GOOD   -0.050017  0.531044  1.000000  0.302799  0.511265
TIME_Q -0.121962  0.459284  0.302799  1.000000  0.272769
PAY    -0.042782  0.506574  0.511265  0.272769  1.000000


In [18]:
# find correlations between reward columns
reward_columns = ['likedness', 'difficulty', 'time_spent', 'usefulness', 'PAY_next', 'completed']
correlation_matrix = df_final[reward_columns].corr()
print(correlation_matrix)

            likedness  difficulty  time_spent  usefulness  PAY_next  completed
likedness    1.000000   -0.049535    0.152128    0.646439  0.158477        NaN
difficulty  -0.049535    1.000000    0.108237    0.066781  0.020921        NaN
time_spent   0.152128    0.108237    1.000000    0.137441  0.007789        NaN
usefulness   0.646439    0.066781    0.137441    1.000000  0.243437        NaN
PAY_next     0.158477    0.020921    0.007789    0.243437  1.000000    0.08752
completed         NaN         NaN         NaN         NaN  0.087520    1.00000
